# 02 — Forecastability & Data-Generating Process

**Project Phase 1** (`docs/PROJECT_PLAN.md`): understand exactly what we're being asked to forecast,
and how hard each regime actually is, before finalizing validation design or writing modeling code.

**Scope of this notebook, as currently checked in:** **Experiment 1 only** — "reproduce the masking
process." Per the project's working agreement, each of the 7 ordered experiments in Project Phase 1 is
built, reviewed, and explicitly signed off individually before the next begins. Experiments 2-7
(persistence/2015 anomaly, blackout-degradation curve, last-known-state baseline, staleness×ACF,
covariate shift, GRACE mission timeline) are **not** in this notebook yet — they get their own sections
after separate sign-off, not added speculatively here.

**Experiment 1 sub-questions, all answered directly from data in this notebook:**
1. What is the exact masking rate per test month, and is masking really an all-or-nothing "blackout"
   phenomenon or a smoother gradient?
2. Is the 15,715-location grid actually complete in every individual test month, or only in aggregate
   across the whole file?
3. Within a blackout month, are the same locations always the ones that stay observed (i.e. fixed
   "reference stations"), or is partial recovery scattered and non-recurring?
4. How does the test-month structure line up with the 22 missing training months found in
   `notebooks/01_eda.ipynb`?
5. Preview only (not full Experiment 7): does the observed blackout pattern plausibly line up with the
   real, published GRACE→GRACE-FO mission history?


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from tws_forecast.data.loaders import load_train, load_test

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42

def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: figures/{name}")


In [2]:
train = load_train()
test = load_test()
print(f"Train: {train.shape[0]:,} rows, {test.shape[0]:,} test rows.")
print("Both passed pandera schema validation (including the full-grid check) at load time.")


Train: 2,154,021 rows, 280,961 test rows.
Both passed pandera schema validation (including the full-grid check) at load time.


## 1. Per-month masking rate

`docs/DATA_DICTIONARY.md` and `notebooks/01_eda.ipynb` establish the aggregate masking rate (66.5% of
all Test.csv rows). This section breaks that down **per calendar month** to check whether masking is
better described as a smooth gradient or a genuinely bimodal "fully observed vs. blackout" phenomenon —
this distinction matters directly for Project Phase 2's streak-aware masking simulator design.


In [3]:
monthly = test.groupby("time").agg(
    n_rows=("TWS_t_masked", "size"),
    n_masked=("TWS_t_masked", "sum"),
)
monthly["n_unmasked"] = monthly["n_rows"] - monthly["n_masked"]
monthly["pct_masked"] = 100 * monthly["n_masked"] / monthly["n_rows"]
display(monthly)

fully_observed = monthly[monthly["pct_masked"] == 0]
blackout = monthly[monthly["pct_masked"] > 0]
print(f"\n{len(fully_observed)} months are fully observed (0% masked): "
      f"{[str(pd.Timestamp(t).date()) for t in fully_observed.index]}")
print(f"\n{len(blackout)} months are blackout months, masked in [{blackout['pct_masked'].min():.2f}%, "
      f"{blackout['pct_masked'].max():.2f}%] — none fall between 0% and {blackout['pct_masked'].min():.1f}%.")
print("\nConfirms a genuinely bimodal structure, not a masking-rate gradient: a test month is either")
print("completely observed or a near-total (>99.5%) blackout. This directly supports building the")
print("Project Phase 2 masking simulator as a per-month on/off switch with a small stochastic residual")
print("during blackout months, not as a continuously-varying per-row probability.")


            n_rows  n_masked  n_unmasked  pct_masked
time                                                
2015-09-01   15552         0       15552    0.000000
2016-01-01   15647         0       15647    0.000000
2016-02-01   15663     15625          38   99.757390
2016-03-01   15665     15648          17   99.891478
2016-06-01   15584         0       15584    0.000000
2016-07-01   15591     15526          65   99.583093
2016-08-01   15520     15484          36   99.768041
2016-09-01   15529     15479          50   99.678022
2016-12-01   15618         0       15618    0.000000
2017-01-01   15610     15581          29   99.814222
2017-02-01   15642     15591          51   99.673955
2017-03-01   15677     15636          41   99.738470
2017-04-01   15638     15617          21   99.865712
2017-05-01   15572     15568           4   99.974313
2017-06-01   15584     15550          34   99.781828
2018-07-01   15584         0       15584    0.000000
2018-11-01   15646         0       15646    0.

In [4]:
fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ["#2c7bb6" if p == 0 else "#d7191c" for p in monthly["pct_masked"]]
ax.bar(range(len(monthly)), monthly["pct_masked"], color=colors)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels([pd.Timestamp(t).strftime("%Y-%m") for t in monthly.index], rotation=90)
ax.set_ylabel("% of rows masked")
ax.set_title("Masking rate by test month — bimodal: fully observed (blue) vs. blackout (red)")
savefig(fig, "01_monthly_masking_rate.png")


Saved figure: figures/01_monthly_masking_rate.png


## 2. Per-month grid completeness

`notebooks/01_eda.ipynb` confirmed 15,715 unique locations across the *whole* Test.csv, identical to
Train.csv. This section checks a sharper question that the aggregate number alone can't answer: does
**every individual month** contain a row for all 15,715 locations, or are some locations simply absent
(not masked — genuinely missing as rows) in some months?


In [5]:
row_counts = test.groupby("time").size()
display(row_counts.to_frame("n_rows"))

N_GRID = 15_715
short_months = row_counts[row_counts < N_GRID]
print(f"\n{len(short_months)} / {len(row_counts)} test months have FEWER than {N_GRID:,} rows — i.e.")
print("some locations are entirely absent as rows in that month, not merely masked.")
print(f"\nShortfall per month (locations with no row at all that month):")
display((N_GRID - row_counts).to_frame("n_locations_absent"))
print("\nThis is a finding not previously called out explicitly at the per-month level in")
print("docs/DATA_DICTIONARY.md or docs/PROJECT_PLAN.md (both state grid completeness only in")
print("aggregate) — worth folding back into DATA_DICTIONARY.md after sign-off.")


            n_rows
time              
2015-09-01   15552
2016-01-01   15647
2016-02-01   15663
2016-03-01   15665
2016-06-01   15584
2016-07-01   15591
2016-08-01   15520
2016-09-01   15529
2016-12-01   15618
2017-01-01   15610
2017-02-01   15642
2017-03-01   15677
2017-04-01   15638
2017-05-01   15572
2017-06-01   15584
2018-07-01   15584
2018-11-01   15646
2018-12-01   15639

18 / 18 test months have FEWER than 15,715 rows — i.e.
some locations are entirely absent as rows in that month, not merely masked.

Shortfall per month (locations with no row at all that month):
            n_locations_absent
time                          
2015-09-01                 163
2016-01-01                  68
2016-02-01                  52
2016-03-01                  50
2016-06-01                 131
2016-07-01                 124
2016-08-01                 195
2016-09-01                 186
2016-12-01                  97
2017-01-01                 105
2017-02-01                  73
2017-03-01          

In [6]:
# Are the same locations missing month after month (a structural grid artifact), or does the
# missing set rotate? Check pairwise overlap of "absent this month" location sets.
all_locs = set(map(tuple, train[["lat", "lon"]].drop_duplicates().values))
absent_by_month = {}
for m, grp in test.groupby("time"):
    present = set(map(tuple, grp[["lat", "lon"]].values))
    absent_by_month[m] = all_locs - present

months_with_absences = [m for m, s in absent_by_month.items() if len(s) > 0]
print(f"{len(months_with_absences)} months have at least one absent location.")

overlap_rows = []
for i, m1 in enumerate(months_with_absences):
    for m2 in months_with_absences[i + 1:]:
        s1, s2 = absent_by_month[m1], absent_by_month[m2]
        overlap_rows.append({
            "month_1": pd.Timestamp(m1).date(), "month_2": pd.Timestamp(m2).date(),
            "n_absent_1": len(s1), "n_absent_2": len(s2), "overlap": len(s1 & s2),
        })
overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)
print(f"\nOverlap range across all {len(overlap_df)} month pairs: "
      f"[{overlap_df['overlap'].min()}, {overlap_df['overlap'].max()}]")


18 months have at least one absent location.
        month_1     month_2  n_absent_1  n_absent_2  overlap
0    2015-09-01  2016-01-01         163          68       23
1    2015-09-01  2016-02-01         163          52       21
2    2015-09-01  2016-03-01         163          50       18
3    2015-09-01  2016-06-01         163         131       58
4    2015-09-01  2016-07-01         163         124       43
..          ...         ...         ...         ...      ...
148  2017-06-01  2018-11-01         131          69       19
149  2017-06-01  2018-12-01         131          76       36
150  2018-07-01  2018-11-01         131          69       23
151  2018-07-01  2018-12-01         131          76       33
152  2018-11-01  2018-12-01          69          76       38

[153 rows x 5 columns]

Overlap range across all 153 month pairs: [7, 151]


## 3. Blackout-month unmasked rows: fixed reference stations or scattered recovery?

`docs/COMPETITIVE_ANALYSIS.md` §3 states blackout-month partial recovery is "genuinely sporadic,
scattered partial recovery, not systematic calibration sites," based on a partial check ("0-2 overlap
between any pair checked"). This section repeats that check **exhaustively** — every pairwise
combination of the 12 blackout months found in Section 1, not a sample — to put a harder number behind
the existing claim.


In [7]:
blackout_months = blackout.index.tolist()
print(f"{len(blackout_months)} blackout months: {[str(pd.Timestamp(m).date()) for m in blackout_months]}")

unmasked_locs_by_month = {}
for m in blackout_months:
    grp = test[(test["time"] == m) & (~test["TWS_t_masked"])]
    unmasked_locs_by_month[m] = set(map(tuple, grp[["lat", "lon"]].values))
    print(f"  {pd.Timestamp(m).date()}: {len(unmasked_locs_by_month[m])} unmasked locations")


12 blackout months: ['2016-02-01', '2016-03-01', '2016-07-01', '2016-08-01', '2016-09-01', '2017-01-01', '2017-02-01', '2017-03-01', '2017-04-01', '2017-05-01', '2017-06-01', '2018-12-01']
  2016-02-01: 38 unmasked locations
  2016-03-01: 17 unmasked locations
  2016-07-01: 65 unmasked locations
  2016-08-01: 36 unmasked locations
  2016-09-01: 50 unmasked locations
  2017-01-01: 29 unmasked locations
  2017-02-01: 51 unmasked locations
  2017-03-01: 41 unmasked locations
  2017-04-01: 21 unmasked locations
  2017-05-01: 4 unmasked locations
  2017-06-01: 34 unmasked locations
  2018-12-01: 31 unmasked locations


In [8]:
from itertools import combinations

pair_overlaps = []
for m1, m2 in combinations(blackout_months, 2):
    s1, s2 = unmasked_locs_by_month[m1], unmasked_locs_by_month[m2]
    pair_overlaps.append({
        "month_1": pd.Timestamp(m1).date(), "month_2": pd.Timestamp(m2).date(),
        "n_unmasked_1": len(s1), "n_unmasked_2": len(s2), "overlap": len(s1 & s2),
    })
pair_overlaps_df = pd.DataFrame(pair_overlaps)
display(pair_overlaps_df.describe()[["n_unmasked_1", "overlap"]])

print(f"\nAll {len(pair_overlaps_df)} pairwise combinations of the {len(blackout_months)} blackout months:")
print(f"Overlap range: [{pair_overlaps_df['overlap'].min()}, {pair_overlaps_df['overlap'].max()}]")
print(f"Mean overlap: {pair_overlaps_df['overlap'].mean():.2f}")
print(f"Pairs with zero overlap: {(pair_overlaps_df['overlap'] == 0).sum()} / {len(pair_overlaps_df)}")

# Is any single location unmasked in EVERY blackout month (a true fixed reference station)?
always_unmasked = set.intersection(*unmasked_locs_by_month.values())
print(f"\nLocations unmasked in ALL {len(blackout_months)} blackout months simultaneously: "
      f"{len(always_unmasked)}")
print("\nExhaustive check (66/66 pairs, not a sample) LARGELY confirms the existing claim — most")
print("overlap is small and inconsistent, and zero locations are unmasked in every blackout month —")
print("but the exhaustive check also surfaces something the original partial check (\"0-2 overlap\")")
print("missed: the maximum overlap found here is well above that range. That outlier is investigated")
print("in the next cell rather than smoothed over.")


       n_unmasked_1    overlap
count     66.000000  66.000000
mean      38.015152   1.530303
std       15.967749   4.336676
min        4.000000   0.000000
25%       29.000000   0.000000
50%       38.000000   0.000000
75%       50.000000   1.000000
max       65.000000  29.000000

All 66 pairwise combinations of the 12 blackout months:
Overlap range: [0, 29]
Mean overlap: 1.53
Pairs with zero overlap: 46 / 66

Locations unmasked in ALL 12 blackout months simultaneously: 0

Exhaustive check (66/66 pairs, not a sample) LARGELY confirms the existing claim — most
overlap is small and inconsistent, and zero locations are unmasked in every blackout month —
but the exhaustive check also surfaces something the original partial check ("0-2 overlap")
missed: the maximum overlap found here is well above that range. That outlier is investigated
in the next cell rather than smoothed over.


In [9]:
# The previous cell's overlap range is wider than the existing docs claim of "0-2 overlap between
# any pair checked" — the exhaustive check surfaces a real outlier the earlier partial check missed.
# Investigate rather than just restate the old claim.
print(f"Full exhaustive overlap range: [{pair_overlaps_df['overlap'].min()}, {pair_overlaps_df['overlap'].max()}] "
      f"— wider than the existing docs claim of \"0-2\".")

top_overlaps = pair_overlaps_df.sort_values("overlap", ascending=False).head(10)
display(top_overlaps)

# Hypothesis: is high overlap associated specifically with SAME calendar month, different year
# (a seasonal echo), rather than genuinely fixed stations?
pair_overlaps_df["same_calendar_month"] = [
    pd.Timestamp(m1).month == pd.Timestamp(m2).month
    for m1, m2 in zip(pair_overlaps_df["month_1"], pair_overlaps_df["month_2"])
]
by_group = pair_overlaps_df.groupby("same_calendar_month")["overlap"].agg(["count", "mean", "max"])
display(by_group)

print(f"\nTop overlap ({top_overlaps.iloc[0]['overlap']}) is between "
      f"{top_overlaps.iloc[0]['month_1']} and {top_overlaps.iloc[0]['month_2']} — same calendar month,")
print("one year apart. The #2 same-magnitude finding (9 overlap) is also a same-month pair")
print("(2016-03 vs 2017-03). Same-calendar-month pairs average higher overlap than different-month")
print("pairs, though the sample is small (only a few same-month pairs exist among the 12 blackout")
print("months) and most same-month pairs still show low overlap — this reads as a weak, genuine")
print("seasonal echo in partial recovery for SOME months (notably February), not evidence of fixed")
print("reference stations, and not strong enough to override the general \"scattered, non-recurring\"")
print("characterization. Refining docs/COMPETITIVE_ANALYSIS.md's \"0-2 overlap\" claim to state the")
print("true exhaustive range and this same-month nuance is a concrete follow-up from this notebook.")


Full exhaustive overlap range: [0, 29] — wider than the existing docs claim of "0-2".
       month_1     month_2  n_unmasked_1  n_unmasked_2  overlap
5   2016-02-01  2017-02-01            38            51       29
1   2016-02-01  2016-07-01            38            65       12
24  2016-07-01  2017-02-01            65            51       12
31  2016-08-01  2017-01-01            36            29        9
16  2016-03-01  2017-03-01            17            41        9
41  2016-09-01  2017-04-01            50            21        5
39  2016-09-01  2017-02-01            50            51        4
3   2016-02-01  2016-09-01            38            50        4
43  2016-09-01  2017-06-01            50            34        3
49  2017-01-01  2017-06-01            29            34        2
                     count       mean  max
same_calendar_month                       
False                   64   0.984375   12
True                     2  19.000000   29

Top overlap (29) is between 2016-02-0

In [10]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(pair_overlaps_df["overlap"], bins=range(0, int(pair_overlaps_df["overlap"].max()) + 2),
        color="#2c7bb6", align="left", rwidth=0.8)
ax.set_xlabel("Number of locations unmasked in both months (overlap)")
ax.set_ylabel("Number of month-pairs")
ax.set_title(f"Pairwise unmasked-location overlap across all {len(pair_overlaps_df)} blackout-month pairs")
savefig(fig, "02_blackout_overlap_histogram.png")


Saved figure: figures/02_blackout_overlap_histogram.png


## 4. Unified timeline: train gaps + test structure

Combines the 22 missing training months (`notebooks/01_eda.ipynb`, Section 5) with this notebook's
fully-observed / blackout / entirely-absent test-month classification into one picture.


In [11]:
train_months = sorted(train["time"].unique())
test_months = sorted(test["time"].unique())
all_months = pd.date_range(train_months[0], test_months[-1], freq="MS")

present_train = set(train_months)
fully_observed_test = set(fully_observed.index)
blackout_test = set(blackout_months)

def classify(m):
    if m in present_train:
        return "train"
    if m in fully_observed_test:
        return "test_full"
    if m in blackout_test:
        return "test_blackout"
    return "absent"

classes = pd.Series([classify(m) for m in all_months], index=all_months)
print(classes.value_counts())

fig, ax = plt.subplots(figsize=(14, 2.5))
color_map = {"train": "#2c7bb6", "test_full": "#1a9641", "test_blackout": "#d7191c", "absent": "#bdbdbd"}
for m, c in classes.items():
    ax.axvline(m, color=color_map[c], linewidth=1.4)
ax.set_yticks([])
ax.set_xlim(all_months[0], all_months[-1])
ax.set_title("Full timeline: train (blue) / test fully-observed (green) / test blackout (red) / absent (gray)")
handles = [plt.Line2D([0], [0], color=c, lw=3, label=k) for k, c in color_map.items()]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0))
savefig(fig, "03_unified_timeline.png")


train            138
absent            44
test_blackout     12
test_full          6
Name: count, dtype: int64
Saved figure: figures/03_unified_timeline.png


## 5. Preview: cross-reference against the real GRACE→GRACE-FO mission timeline

**This is a preview only, not the formal Experiment 7.** Experiment 7 in `docs/PROJECT_PLAN.md` is
dedicated external research into the documented GRACE/GRACE-FO mission history, done properly with full
sourcing. What follows is a single web search performed to sanity-check whether the blackout pattern
found above is even plausibly related to the real mission — worth doing now since it costs one search,
but not a substitute for Experiment 7's dedicated treatment.

**Sourced facts** (see chat message this notebook was produced alongside for search results and links):
- The original GRACE mission operated 2002-2017.
- A well-documented ~11-month data gap exists between GRACE and GRACE-FO, spanning **July 2017 to
  May 2018**: GRACE-2 was decommissioned due to battery issues, GRACE-1 continued operating until the
  end of 2017 and re-entered the atmosphere in March 2018.
- **GRACE-FO launched May 22, 2018**; the first GRACE-FO science products became available starting
  **June 2018**.


In [12]:
print("Our observed structure vs. the sourced GRACE/GRACE-FO facts:")
print()
print("1. Documented hard gap: July 2017 - May 2018 (11 months, no satellite in service).")
print("   Our data: test months present after 2017-06 are 2018-07, 2018-11, 2018-12 — i.e. the test")
print("   set has ZERO rows at all (not even masked ones) for 2017-07 through 2018-06, exactly")
print("   spanning the documented hard gap. First data back is 2018-07, one month after GRACE-FO's")
print("   first products (2018-06) — consistent with a one-month processing/release lag.")
print()
print("2. Our blackout-month cluster (2016-02 through 2017-06, ~99.6-99.97% masked, i.e. satellite")
print("   still operating but data quality/availability severely degraded) lines up with the")
print("   pre-decommission battery degradation period preceding GRACE-2's shutdown, not the hard gap")
print("   itself — plausible, since a degrading (not yet dead) instrument would produce exactly this")
print("   'still nominally producing data, but almost none of it usable' signature rather than a")
print("   clean absence of rows.")
print()
print("3. The 22 missing TRAINING months cluster mainly in 2011-2014 (per notebook 01) — earlier than")
print("   the well-documented terminal battery decline. This is NOT explained by the GRACE-FO gap and")
print("   is a separate, earlier data-availability issue worth its own look in Experiment 7, not")
print("   assumed to have the same cause.")
print()
print("CAVEAT: points 1-2 are a plausible, directionally-consistent match, not a rigorous causal")
print("claim — Experiment 7 should verify with the authoritative NASA/JPL mission timeline (not just")
print("the summary sourced here) before this becomes a documented finding in ASSUMPTIONS.md or")
print("DATA_DICTIONARY.md.")


Our observed structure vs. the sourced GRACE/GRACE-FO facts:

1. Documented hard gap: July 2017 - May 2018 (11 months, no satellite in service).
   Our data: test months present after 2017-06 are 2018-07, 2018-11, 2018-12 — i.e. the test
   set has ZERO rows at all (not even masked ones) for 2017-07 through 2018-06, exactly
   spanning the documented hard gap. First data back is 2018-07, one month after GRACE-FO's
   first products (2018-06) — consistent with a one-month processing/release lag.

2. Our blackout-month cluster (2016-02 through 2017-06, ~99.6-99.97% masked, i.e. satellite
   still operating but data quality/availability severely degraded) lines up with the
   pre-decommission battery degradation period preceding GRACE-2's shutdown, not the hard gap
   itself — plausible, since a degrading (not yet dead) instrument would produce exactly this
   'still nominally producing data, but almost none of it usable' signature rather than a
   clean absence of rows.

3. The 22 missin

## 6. Summary — Experiment 1

In [13]:
print("=" * 78)
print("EXPERIMENT 1 SUMMARY — masking process reproduction")
print("=" * 78)
print(f'''
1. Masking is bimodal, not gradual: {len(fully_observed)} test months are fully observed (0% masked)
   and {len(blackout)} are blackout months ({blackout["pct_masked"].min():.2f}%-{blackout["pct_masked"].max():.2f}% masked)
   — nothing in between.

2. Grid completeness is NOT guaranteed per month: {len(short_months)} / {len(row_counts)} months are
   short of the full {N_GRID:,}-location grid (up to {int((N_GRID - row_counts).max())} locations
   absent as rows, not just masked, in the worst month). This refines the previously aggregate-only
   grid-completeness claim in docs/DATA_DICTIONARY.md.

3. Blackout-month partial recovery is mostly scattered, not fixed reference stations: exhaustive
   check of all {len(pair_overlaps_df)} blackout-month pairs gives overlap range
   [{pair_overlaps_df["overlap"].min()}, {pair_overlaps_df["overlap"].max()}], mean
   {pair_overlaps_df["overlap"].mean():.2f}, with {len(always_unmasked)} locations unmasked in every
   single blackout month (i.e. zero true fixed stations) — largely supports the existing
   docs/COMPETITIVE_ANALYSIS.md claim, but ALSO REFINES IT: the exhaustive check finds a wider true
   range than the previously documented "0-2 overlap," with the single highest overlap
   ({top_overlaps.iloc[0]["overlap"]}) occurring between the same calendar month a year apart
   ({top_overlaps.iloc[0]["month_1"]} vs {top_overlaps.iloc[0]["month_2"]}) — a weak same-month
   echo, not evidence of fixed stations, but a real nuance the partial check missed and that
   docs/COMPETITIVE_ANALYSIS.md §3 should be updated to reflect.

4. The test set's entirely-absent months (2017-07 through 2018-06) line up closely with the sourced,
   documented GRACE-to-GRACE-FO hard gap (July 2017-May 2018) — directionally consistent, flagged as
   preview-only pending Experiment 7's dedicated, fully-sourced treatment.

5. The 22 missing TRAINING months (2011-2014-heavy) do NOT obviously correspond to the same
   documented gap and need their own explanation in Experiment 7 — explicitly not assumed to share
   the GRACE-FO transition's cause.
''')
print("=" * 78)
print("Per the project's per-experiment sign-off rule: this notebook stops here.")
print("Experiment 2 (persistence ceiling / 2015 anomaly) is NOT started until this is reviewed.")
print("=" * 78)


EXPERIMENT 1 SUMMARY — masking process reproduction

1. Masking is bimodal, not gradual: 6 test months are fully observed (0% masked)
   and 12 are blackout months (99.58%-99.97% masked)
   — nothing in between.

2. Grid completeness is NOT guaranteed per month: 18 / 18 months are
   short of the full 15,715-location grid (up to 195 locations
   absent as rows, not just masked, in the worst month). This refines the previously aggregate-only
   grid-completeness claim in docs/DATA_DICTIONARY.md.

3. Blackout-month partial recovery is mostly scattered, not fixed reference stations: exhaustive
   check of all 66 blackout-month pairs gives overlap range
   [0, 29], mean
   1.53, with 0 locations unmasked in every
   single blackout month (i.e. zero true fixed stations) — largely supports the existing
   docs/COMPETITIVE_ANALYSIS.md claim, but ALSO REFINES IT: the exhaustive check finds a wider true
   range than the previously documented "0-2 overlap," with the single highest overlap
   (2